# Bab 3 — Introduction to neural prediction: forward propagation

**Mahasiswa:** Tarannia Raisha Putri · 101032300231 · TK4702 · Deep Learning

**Referensi utama:** Andrew W. Trask, *Grokking Deep Learning*, Manning (2019), Bab 3, halaman cetak 21–46 (PDF 42–67, indeks PDF mulai 1). Subbab spesifik dicantumkan pada setiap tahap.

**Tujuan pembelajaran:** Mengimplementasikan weighted sum, vector–matrix multiplication, dua lapis linear, serta mendiagnosis bentuk array.

## Ringkasan

Forward propagation mengubah input menjadi prediksi dengan perkalian dan penjumlahan berbobot. Alur buku bergerak dari satu input dan satu output menuju banyak input/output, kemudian prediksi bertingkat dan operasi NumPy.

## Konsep, intuisi, dan notasi

Untuk satu input, $\hat y=xw$ (skalar). Banyak input: $\hat y=\sum_{i=1}^d x_iw_i=x^Tw$. Banyak output dengan konvensi vektor baris: $\hat y=xW$, $x\in\mathbb R^{1\times d}$, $W\in\mathbb R^{d\times k}$, $\hat y\in\mathbb R^{1\times k}$.

Listing loop buku menyimpan satu vektor bobot per output: bentuknya $(k,d)$. Karena itu versi NumPy memerlukan transpose. Transpose bukan trik menghindari error; ia menyesuaikan definisi sumbu.

Dua lapis linear menghasilkan $h=xW_0$, $\hat y=hW_1$. Tanpa nonlinearity, hasilnya setara dengan $x(W_0W_1)$. Bobot positif meneruskan kontribusi input searah, bobot negatif membaliknya, bobot nol mengabaikannya. Nilai bobot dan satuan fitur sama-sama menentukan kontribusi. Angka output belum otomatis probabilitas dan belum ada training.

## Persiapan dan cara membaca

Jalankan **Restart Kernel → Run All**. Semua penjelasan merupakan parafrasa orisinal; angka hasil berada pada output sel yang benar-benar dijalankan. Label **reproduksi** berarti algoritme/contoh buku ditulis ulang; **koreksi/penyesuaian** menunjukkan perubahan; **tambahan** menunjukkan eksperimen atau interpretasi praktikum. `GDL_MODE=quick` mengurangi latihan berat; `full` memilih anggaran buku, dengan perubahan yang tetap dicatat. Keberhasilan eksekusi tidak membuktikan kesamaan akurasi dengan buku.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Jalankan dari root repository atau notebooks/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MODE = os.environ.get('GDL_MODE', 'quick')
assert MODE in {'quick', 'full'}
FULL = MODE == 'full'
np.random.seed(1)
np.set_printoptions(precision=6, suppress=True)
METRICS = {}
def record(**values):
    METRICS.update(values)
def finite(*arrays):
    assert all(np.isfinite(np.asarray(a)).all() for a in arrays), 'NaN/inf terdeteksi'
def softmax(z):
    z = np.asarray(z)
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)
def sigmoid(z):
    return np.exp(-np.logaddexp(0, -np.asarray(z)))
print('Mode:', MODE, '| NumPy:', np.__version__)

Mode: quick | NumPy: 2.5.4


<a id="b03-scalar"></a>

## Reproduksi: satu bobot

**Sumber:** cetak 22–27; PDF 43–48.

Data jumlah jari kaki dan bobot merupakan contoh olahraga ilustratif dari buku, bukan dataset pertandingan nyata. Fungsi hanya menghitung prediksi; belum menggunakan target untuk memperbarui parameter.

In [2]:
number_of_toes = [8.5, 9.5, 10, 9]
def neural_prediction(x, weight):
    return x * weight
pred = neural_prediction(number_of_toes[0], 0.1)
print('Prediksi satu input:', pred)
assert np.isclose(pred, 0.85)

Prediksi satu input: 0.8500000000000001


<a id="b03-inputs"></a>

## Reproduksi: banyak input dan versi NumPy

**Sumber:** cetak 28–35; PDF 49–56.

Weighted sum mengalikan fitur dengan koefisien yang sesuai sebelum menjumlahkan. `fans` berunit juta, sehingga mengganti unit tanpa menyesuaikan bobot mengubah prediksi. Loop dan `dot` diperiksa kesetaraannya.

In [3]:
def weighted_sum(a, b):
    assert len(a) == len(b)
    total = 0.0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total
X = np.array([[8.5,.65,1.2],[9.5,.8,1.3],[9.9,.8,.5],[9.,.9,1.]])
w = np.array([.1,.2,0.])
manual = weighted_sum(X[0], w)
print('Loop:', manual, 'NumPy:', X[0] @ w)
assert np.isclose(manual, .98)
assert np.isclose(manual, X[0] @ w)

Loop: 0.9800000000000001 NumPy: 0.9800000000000001


<a id="b03-outputs"></a>

## Reproduksi: banyak output, kemudian banyak input-output

**Sumber:** cetak 36–41; PDF 57–62.

Satu input dapat dikalikan dengan tiga bobot untuk tiga prediksi. Untuk banyak input-output, setiap output mempunyai weighted sum sendiri. Loop digeneralisasi agar jumlah input tidak harus sama dengan jumlah output; pembatasan ukuran 3 pada contoh buku tidak diperlukan.

In [4]:
def scalar_vector_mul(x, weights):
    return [x*w for w in weights]
print('Satu input, tiga output:', scalar_vector_mul(.65,[.3,.2,.9]))
W_rows = np.array([[.1,.1,-.3],[.1,.2,0.],[0.,1.3,.1]])
def vector_matrix_rows(x, rows):
    return np.array([weighted_sum(x, row) for row in rows])
pred = vector_matrix_rows(X[0], W_rows)
print('Banyak input-output:', pred)
np.testing.assert_allclose(pred, [.555,.98,.965])
np.testing.assert_allclose(pred, X[0] @ W_rows.T)

Satu input, tiga output: [0.195, 0.13, 0.5850000000000001]
Banyak input-output: [0.555 0.98  0.965]


<a id="b03-stack"></a>

## Reproduksi: predicting on predictions

**Sumber:** cetak 42–43; PDF 63–64.

Hidden layer berisi prediksi antara yang dipakai sebagai fitur bagi lapisan berikutnya. Tidak ada aktivasi nonlinear pada bab ini. Eksperimen penggabungan bobot di bawah adalah tambahan untuk menunjukkan batas representasinya.

In [5]:
W0_rows = np.array([[.1,.2,-.1],[-.1,.1,.9],[.1,.4,.1]])
W1_rows = np.array([[.3,1.1,-.3],[.1,.2,0.],[0.,1.3,.1]])
hidden = vector_matrix_rows(X[0], W0_rows)
pred = vector_matrix_rows(hidden, W1_rows)
print('Hidden:', hidden, '| output:', pred)
np.testing.assert_allclose(pred, X[0] @ W0_rows.T @ W1_rows.T)
np.testing.assert_allclose(pred, X[0] @ (W0_rows.T @ W1_rows.T))
record(scalar_prediction=.85, multiple_input_prediction=manual, stacked_output=pred.tolist())

Hidden: [0.86  0.295 1.23 ] | output: [0.2135 0.145  0.5065]


<a id="b03-numpy"></a>

## Reproduksi: primer NumPy dan kegagalan bentuk

**Sumber:** cetak 44–46; PDF 65–67.

`*` adalah perkalian elemen dengan broadcasting; `@` menjumlahkan hasil perkalian sepanjang sumbu dalam. Error yang sengaja dicontohkan buku ditangkap agar Run All tetap selesai; pesan error asli tetap ditampilkan.

In [6]:
a=np.array([0,1,2,3]); b=np.array([4,5,6,7]); c=np.array([[0,1,2,3],[4,5,6,7]])
print('Array dan shape:',a,a.shape,c,c.shape)
print('Skala:',a*.1,c*.2, '| elemen:',a*b, '| broadcast:',a*c)
print('Zeros:',np.zeros((2,4)), '| random:',np.random.rand(2,5))
for A,B in [(np.zeros((1,4)),np.zeros((4,3))), (np.zeros((2,4)),np.zeros((4,3))),
            (np.zeros((2,1)),np.zeros((1,3))), (np.zeros((5,4)).T,np.zeros((5,6)))]:
    print(A.shape,'@',B.shape,'->',(A@B).shape)
for operation in [lambda: a*np.zeros((2,5)), lambda: np.zeros((5,4))@np.zeros((5,6))]:
    try: operation()
    except ValueError as exc: print('Kegagalan yang diharapkan:',str(exc).splitlines()[0])
    else: raise AssertionError('Contoh bentuk salah seharusnya gagal')

Array dan shape: [0 1 2 3] (4,) [[0 1 2 3]
 [4 5 6 7]] (2, 4)
Skala: [0.  0.1 0.2 0.3] [[0.  0.2 0.4 0.6]
 [0.8 1.  1.2 1.4]] | elemen: [ 0  5 12 21] | broadcast: [[ 0  1  4  9]
 [ 0  5 12 21]]
Zeros: [[0. 0. 0. 0.]
 [0. 0. 0. 0.]] | random: [[0.417022 0.720324 0.000114 0.302333 0.146756]
 [0.092339 0.18626  0.345561 0.396767 0.538817]]
(1, 4) @ (4, 3) -> (1, 3)
(2, 4) @ (4, 3) -> (2, 3)
(2, 1) @ (1, 3) -> (2, 3)
(4, 5) @ (5, 6) -> (4, 6)
Kegagalan yang diharapkan: operands could not be broadcast together with shapes (4,) (2,5) 
Kegagalan yang diharapkan: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 5 is different from 4)


## Analisis output aktual — run tersimpan

Mode **quick**, eksekusi 2026-10-16T01:50:10.585588+07:00. Ringkasan ini dibuat dari output validator, bukan angka contoh dalam buku.

Prediksi satu input 0.85; banyak input 0.98. Output jaringan bertingkat [0.21350000000000002, 0.14500000000000002, 0.5065] cocok antara loop dan NumPy. Kesamaan ini menguji operasi forward, bukan kualitas bobot.

## Kesimpulan dan pembahasan pertanyaan

Prediksi loop dan NumPy sama setelah orientasi bobot disamakan. Dua lapis linear dapat digabung menjadi satu matriks; alasan memerlukan nonlinearity akan terlihat pada Bab 6.

**Mengapa transpose digunakan?** Buku mula-mula mengelompokkan bobot menurut output, sedangkan vektor baris NumPy membutuhkan input pada sumbu pertama.

**Apakah output 0,98 berarti peluang 98%?** Belum; model linear tidak membatasi rentang atau menjamin kalibrasi.

**Bisakah 4 input menghasilkan 2 output?** Ya: matriks bobot berukuran 4×2.

In [7]:
finite(*[v for v in METRICS.values() if isinstance(v, (int, float))])
print('VALIDATION_METRICS=' + json.dumps(METRICS, sort_keys=True))

VALIDATION_METRICS={"multiple_input_prediction": 0.9800000000000001, "scalar_prediction": 0.85, "stacked_output": [0.21350000000000002, 0.14500000000000002, 0.5065]}
